### FASE 3 da Libertadores - Quartas de Final

In [1]:
import cartolafc
import pandas as pd
from difflib import get_close_matches
import json
from pathlib import Path

pd.set_option('display.max_columns', 50)            # permite a visualização de 50 colunas do dataframe
pd.options.display.float_format = '{:.2f}'.format   # pandas: para todos os números aparecerem com duas casas decimais

# Cria uma instância da API
api = cartolafc.Api(attempts=5)

2026-04-26 23:11:44,779 - numexpr.utils - INFO - NumExpr defaulting to 8 threads.


🏆 4ªs DE FINAL:

⚽(JG1) 1° X 8°
⚽(JG2) 2° X 7°
⚽(JG3) 3° X 6°
⚽(JG4) 4° X 5°

### Dicionário com os IDs e Nomes dos times.

In [2]:
def carregar_payload_js(caminho_arquivo, nome_constante):
    conteudo = Path(caminho_arquivo).read_text(encoding="utf-8")
    prefixo = f"const {nome_constante} = "
    if prefixo not in conteudo:
        raise ValueError(f"Constante '{nome_constante}' n\u00e3o encontrada em {caminho_arquivo}")

    payload = conteudo.split(prefixo, 1)[1].lstrip()
    delimitador_final = "]" if payload.startswith("[") else "}"
    profundidade = 0
    fim = None

    for i, caractere in enumerate(payload):
        if caractere in "[{":
            profundidade += 1
        elif caractere in "]}":
            profundidade -= 1
            if profundidade == 0 and caractere == delimitador_final:
                fim = i + 1
                break

    if fim is None:
        raise ValueError(f"N\u00e3o foi poss\u00edvel extrair o JSON de {caminho_arquivo}")

    conteudo_json = payload[:fim]
    return json.loads(conteudo_json)


caminho_classificados_fase_2 = Path("times_1o_e_2o_lugar_fase_2.js")
if not caminho_classificados_fase_2.exists():
    caminho_classificados_fase_2 = Path("libertadores") / "datasets_liberta" / "times_1o_e_2o_lugar_fase_2.js"

classificados_fase_2 = carregar_payload_js(
    caminho_classificados_fase_2,
    "timesPrimeiroESegundoLugarFase2"
)

df_classificados_fase_2 = pd.DataFrame(classificados_fase_2).rename(
    columns={
        "origem": "Origem",
        "grupo": "Grupo",
        "posicao": "Posição",
        "id": "ID do Time",
        "nome": "Nome do Time",
        "pontos": "Pontos",
        "vitorias": "Vitórias",
        "empates": "Empates",
        "derrotas": "Derrotas",
        "totalCartola": "Total Cartola",
        "cartolaSofrido": "Cartola Sofrido",
        "saldoCartola": "Saldo Cartola",
    }
)

if "Posição" in df_classificados_fase_2.columns:
    df_classificados_fase_2["Posição"] = pd.to_numeric(
        df_classificados_fase_2["Posição"], errors="coerce"
    ).astype("Int64")

nomes_por_id = {
    int(row["ID do Time"]): row["Nome do Time"]
    for _, row in df_classificados_fase_2.dropna(subset=["ID do Time", "Nome do Time"]).iterrows()
}

ranking_colunas = [
    col for col in [
        "Origem", "Grupo", "Posição", "ID do Time", "Nome do Time", "Total Cartola"
    ] if col in df_classificados_fase_2.columns
]

display(df_classificados_fase_2[ranking_colunas])

,Origem,Grupo,Posição,ID do Time,Nome do Time,Total Cartola
0,1o lugar fase 2,Grupo I,1,13913874,Bandoleros FCS,971.76
1,2o lugar fase 2,Grupo I,2,51010813,LISI GREMISTA,915.44
2,1o lugar fase 2,Grupo J,1,3851966,cartola scheuer17,961.09
3,2o lugar fase 2,Grupo J,2,20696550,Dom Camillo68,923.27
4,1o lugar fase 2,Grupo K,1,479510,TEAM LOPES 99,963.58
5,2o lugar fase 2,Grupo K,2,18642587,Fedato Futebol Clube,964.04
6,1o lugar fase 2,Grupo L,1,44810918,lsauer fc,973.56
7,2o lugar fase 2,Grupo L,2,28741323,Tabajara de Inhaua PB1,887.88


In [3]:
# Ranking geral por Total Cartola para definir os confrontos das quartas:
# JG1: 1º x 8º
# JG2: 2º x 7º
# JG3: 3º x 6º
# JG4: 4º x 5º

df_ranking_quartas = (
    df_classificados_fase_2
    .copy()
    .sort_values(["Total Cartola", "Pontos", "Saldo Cartola"], ascending=[False, False, False])
    .reset_index(drop=True)
)
df_ranking_quartas.insert(0, "Ranking Geral", range(1, len(df_ranking_quartas) + 1))

posicoes_quartas = {
    int(row["Ranking Geral"]): int(row["ID do Time"])
    for _, row in df_ranking_quartas.iterrows()
}

pareamentos_quartas = [
    ("Jogo 1 (JG1)", 1, 8),
    ("Jogo 2 (JG2)", 2, 7),
    ("Jogo 3 (JG3)", 3, 6),
    ("Jogo 4 (JG4)", 4, 5),
]

dados_torneio_quartas = []
for jogo, ranking_a, ranking_b in pareamentos_quartas:
    dados_torneio_quartas.extend([
        (jogo, posicoes_quartas[ranking_a]),
        (jogo, posicoes_quartas[ranking_b]),
    ])

# Criar DataFrame base
df_torneio_quartas = pd.DataFrame(dados_torneio_quartas, columns=["Jogo", "ID do Time"])

# Adicionar Nome do Time usando o dicionário
df_torneio_quartas["Nome do Time"] = df_torneio_quartas["ID do Time"].map(nomes_por_id)

# Adicionar ID no Grupo
df_torneio_quartas["ID no Grupo"] = df_torneio_quartas.groupby("Jogo").cumcount() + 1
df_torneio_quartas["ID no Grupo"] = df_torneio_quartas["ID no Grupo"].astype(str) + "_" + df_torneio_quartas["Jogo"].str[-2]

# Reorganizar colunas
df_torneio_quartas_liberta = df_torneio_quartas[["Jogo", "ID do Time", "Nome do Time", "ID no Grupo"]]

df_liberta_jogo_1 = df_torneio_quartas_liberta[df_torneio_quartas_liberta["Jogo"] == "Jogo 1 (JG1)"]
df_liberta_jogo_2 = df_torneio_quartas_liberta[df_torneio_quartas_liberta["Jogo"] == "Jogo 2 (JG2)"]
df_liberta_jogo_3 = df_torneio_quartas_liberta[df_torneio_quartas_liberta["Jogo"] == "Jogo 3 (JG3)"]
df_liberta_jogo_4 = df_torneio_quartas_liberta[df_torneio_quartas_liberta["Jogo"] == "Jogo 4 (JG4)"]

# Lista de grupos
grupos = {
    "Jogo 1 (JG1)": df_liberta_jogo_1,
    "Jogo 2 (JG2)": df_liberta_jogo_2,
    "Jogo 3 (JG3)": df_liberta_jogo_3,
    "Jogo 4 (JG4)": df_liberta_jogo_4,
}

display(df_ranking_quartas)
display(df_liberta_jogo_1)

,Ranking Geral,Origem,Grupo,Posição,ID do Time,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola
0,1,1o lugar fase 2,Grupo L,1,44810918,lsauer fc,12,4,0,2,973.56,445.83,527.73
1,2,1o lugar fase 2,Grupo I,1,13913874,Bandoleros FCS,9,3,0,3,971.76,459.96,511.80
2,3,2o lugar fase 2,Grupo K,2,18642587,Fedato Futebol Clube,6,2,0,4,964.04,474.19,489.85
3,4,1o lugar fase 2,Grupo K,1,479510,TEAM LOPES 99,18,6,0,0,963.58,450.95,512.63
4,5,1o lugar fase 2,Grupo J,1,3851966,cartola scheuer17,15,5,0,1,961.09,388.10,572.99
5,6,2o lugar fase 2,Grupo J,2,20696550,Dom Camillo68,12,4,0,2,923.27,432.87,490.40
6,7,2o lugar fase 2,Grupo I,2,51010813,LISI GREMISTA,9,3,0,3,915.44,423.63,491.81
7,8,2o lugar fase 2,Grupo L,2,28741323,Tabajara de Inhaua PB1,12,4,0,2,887.88,447.60,440.28


,Jogo,ID do Time,Nome do Time,ID no Grupo
0,Jogo 1 (JG1),44810918,lsauer fc,1_1
1,Jogo 1 (JG1),28741323,Tabajara de Inhaua PB1,2_1


### Definição dos Confrontos das 2 Rodadas da Fase 3 da Libertadores (Rodada 13 A 14)

In [4]:
# Rodada 1 - Fase 3 Libertadores (Equivalente a 13ª rodada da competicao) 
confrontos_1a_rodada = [
    # Jogo 1 (JG1)
    ("Jogo 1 (JG1)", "1_1", "2_1"),    

    # Jogo 2 (JG2)
    ("Jogo 2 (JG2)", "1_2", "2_2"),    

    # Jogo 3 (JG3)
    ("Jogo 3 (JG3)", "1_3", "2_3"),   

    # Jogo 4 (JG4)
    ("Jogo 4 (JG4)", "1_4", "2_4")    
]

# Rodada 2 - Fase 3 Libertadores (Equivalente a 14ª rodada da competicao)
confrontos_2a_rodada = [
    # Jogo 1 (JG1)
    ("Jogo 1 (JG1)", "2_1", "1_1"),
    
    # Jogo 2 (JG2)
    ("Jogo 2 (JG2)", "2_2", "1_2"),    

    # Jogo 3 (JG3)
    ("Jogo 3 (JG3)", "2_3", "1_3"),

    # Jogo 4 (JG4)
    ("Jogo 4 (JG4)", "2_4", "1_4")  
]

In [5]:
# Transformar em DataFrame
df_confrontos = pd.DataFrame(confrontos_1a_rodada, columns=["Grupo", "Mandante_ID", "Visitante_ID"])

# Junta com df_torneio para buscar dados dos mandantes
df_mandantes = df_torneio_quartas_liberta.rename(columns={
    "ID no Grupo": "Mandante_ID",
    "Nome do Time": "Mandante_Nome",
    "ID do Time": "Mandante_ID_Time"
})[["Jogo", "Mandante_ID", "Mandante_Nome", "Mandante_ID_Time"]]

# Junta com df_torneio para buscar dados dos visitantes
df_visitantes = df_torneio_quartas_liberta.rename(columns={
    "ID no Grupo": "Visitante_ID",
    "Nome do Time": "Visitante_Nome",
    "ID do Time": "Visitante_ID_Time"    
})[["Jogo", "Visitante_ID", "Visitante_Nome", "Visitante_ID_Time"]]

display(df_confrontos)

,Grupo,Mandante_ID,Visitante_ID
0,Jogo 1 (JG1),1_1,2_1
1,Jogo 2 (JG2),1_2,2_2
2,Jogo 3 (JG3),1_3,2_3
3,Jogo 4 (JG4),1_4,2_4


In [6]:
# Transformar em DataFrame
df_confrontos = pd.DataFrame(confrontos_1a_rodada, columns=["Jogo", "Mandante_ID", "Visitante_ID"])
df_confrontos["Rodada"] = 13
df_rodada_13 = df_confrontos.merge(df_mandantes, on=["Jogo", "Mandante_ID"])
df_rodada_13 = df_rodada_13.merge(df_visitantes, on=["Jogo", "Visitante_ID"])

# Transformar em DataFrame
df_confrontos_2 = pd.DataFrame(confrontos_2a_rodada, columns=["Jogo", "Mandante_ID", "Visitante_ID"])
df_confrontos_2["Rodada"] = 14
df_rodada_14 = df_confrontos_2.merge(df_mandantes, on=["Jogo", "Mandante_ID"])
df_rodada_14 = df_rodada_14.merge(df_visitantes, on=["Jogo", "Visitante_ID"])

display(df_rodada_13)

,Jogo,Mandante_ID,Visitante_ID,Rodada,Mandante_Nome,Mandante_ID_Time,Visitante_Nome,Visitante_ID_Time
0,Jogo 1 (JG1),1_1,2_1,13,lsauer fc,44810918,Tabajara de Inhaua PB1,28741323
1,Jogo 2 (JG2),1_2,2_2,13,Bandoleros FCS,13913874,LISI GREMISTA,51010813
2,Jogo 3 (JG3),1_3,2_3,13,Fedato Futebol Clube,18642587,Dom Camillo68,20696550
3,Jogo 4 (JG4),1_4,2_4,13,TEAM LOPES 99,479510,cartola scheuer17,3851966


In [7]:
df_rodadas = pd.concat([
    df_rodada_13,
    df_rodada_14
], ignore_index=True)

# Ajustar a numeracao da rodada para refletir as rodadas da fase 3 (13 e 14)
df_rodadas["Rodada"] = df_rodadas["Rodada"]

df_rodadas.to_excel("confrontos_fase_3_libertadores.xlsx", index=False)

# Exibir os confrontos da fase 3 (Quartas de Final)
display(df_rodadas.head(8)) 

,Jogo,Mandante_ID,Visitante_ID,Rodada,Mandante_Nome,Mandante_ID_Time,Visitante_Nome,Visitante_ID_Time
0,Jogo 1 (JG1),1_1,2_1,13,lsauer fc,44810918,Tabajara de Inhaua PB1,28741323
1,Jogo 2 (JG2),1_2,2_2,13,Bandoleros FCS,13913874,LISI GREMISTA,51010813
2,Jogo 3 (JG3),1_3,2_3,13,Fedato Futebol Clube,18642587,Dom Camillo68,20696550
3,Jogo 4 (JG4),1_4,2_4,13,TEAM LOPES 99,479510,cartola scheuer17,3851966
4,Jogo 1 (JG1),2_1,1_1,14,Tabajara de Inhaua PB1,28741323,lsauer fc,44810918
5,Jogo 2 (JG2),2_2,1_2,14,LISI GREMISTA,51010813,Bandoleros FCS,13913874
6,Jogo 3 (JG3),2_3,1_3,14,Dom Camillo68,20696550,Fedato Futebol Clube,18642587
7,Jogo 4 (JG4),2_4,1_4,14,cartola scheuer17,3851966,TEAM LOPES 99,479510


In [8]:
# Criar lista de dicionários no formato desejado
confrontos_js_fase_3 = []

for _, row in df_rodadas.iterrows():
    confronto = {
        "jogo": row["Jogo"],
        "rodada": int(row["Rodada"]),
        "mandante": {
            "id": int(row["Mandante_ID_Time"]),
            "nome": row["Mandante_Nome"]
        },
        "visitante": {
            "id": int(row["Visitante_ID_Time"]),
            "nome": row["Visitante_Nome"]        }
    }
    confrontos_js_fase_3.append(confronto)

# Converter para JSON formatado
json_str = json.dumps(confrontos_js_fase_3, indent=2, ensure_ascii=False)

# Salvar como arquivo JS com uma variável global
with open("confrontos_fase3_libertadores.js", "w", encoding="utf-8") as f:
    f.write("const confrontosFase3 = ")
    f.write(json_str)
    f.write(";")

In [9]:
def exibir_confrontos(df_rodadas, rodada=None, jogo=None):
    """
    Filtra e exibe os confrontos por rodada e/ou jogo.
    
    Parâmetros:
    - df_rodadas: DataFrame com todos os confrontos
    - rodada: número da rodada (int ou None para todas)
    - jogo: nome do jogo (str ou None para todos)
    
    Retorna:
    - DataFrame filtrado com as colunas relevantes
    """
    colunas = ["Rodada", "Jogo", "Mandante_Nome", "Visitante_Nome"]
    df_filtrado = df_rodadas.copy()

    df_filtrado["Rodada"] = df_filtrado["Rodada"].astype(str) + "ª Rodada"    

    if rodada is not None:
        df_filtrado = df_filtrado[df_filtrado["Rodada"] == rodada]

    if jogo is not None:
        df_filtrado = df_filtrado[df_filtrado["Jogo"] == jogo]

    return df_filtrado[colunas].sort_values(by=["Jogo", "Rodada"])

In [10]:
jogo = 2

# Exibir todos os confrontos do Jogo  
display(exibir_confrontos(df_rodadas, jogo= f"Jogo {jogo} (JG{jogo})"))

,Rodada,Jogo,Mandante_Nome,Visitante_Nome
1,13ª Rodada,Jogo 2 (JG2),Bandoleros FCS,LISI GREMISTA
5,14ª Rodada,Jogo 2 (JG2),LISI GREMISTA,Bandoleros FCS


In [11]:
# def campeonato_comecou(api, ids_times):
#     """Verifica se o campeonato já começou observando a pontuação na 1ª rodada."""
#     for time_id in ids_times.values():
#         try:
#             pontuacao = api.time(time_id=time_id, rodada=32).ultima_pontuacao
#             if pontuacao is not None:
#                 return True
#         except cartolafc.errors.CartolaFCError:
#             continue
#     return False

# def obter_pontuacao_por_rodada(api, time_id, rodada_atual):
#     """Obtém a pontuação do time em cada rodada até a rodada atual."""
#     pontuacoes = {}
#     for rodada in range(32, rodada_atual):
#         try:
#             time_rodada = api.time(time_id=time_id, rodada=rodada)
#             pontuacoes[rodada] = time_rodada.ultima_pontuacao
#         except cartolafc.errors.CartolaFCError as e:
#             print(f"Erro ao acessar pontuação da rodada {rodada} para o time {time_id}: {e}")
#             pontuacoes[rodada] = None
#     return pontuacoes


# def gerar_df_pontuacoes(api, ids_times):
#     rodada_atual = api.mercado().rodada_atual
#     total_rodadas = 2    

#     if not campeonato_comecou(api, ids_times):
#         print("📌 O campeonato ainda não começou. Criando estrutura com placeholders.")
#         df = pd.DataFrame(index=ids_times.keys(), columns=[f'Rodada {i}' for i in range(32, total_rodadas + 1)])
#         df[:] = 0
#     else:
#         df = pd.DataFrame()
#         for nome, time_id in ids_times.items():
#             pontuacoes = obter_pontuacao_por_rodada(api, time_id, rodada_atual)
#             df[nome] = pd.Series(pontuacoes)
#         df = df.transpose()
#         df.columns = [f'Rodada {i}' for i in range(32, rodada_atual)]
#         df.loc['Lider_Rodada'] = df.idxmax()
    
#     return df



In [12]:
import requests
import time
import sys

FASE_INICIO = 13
FASE_LIMITE = 14
PER_REQ_SLEEP = 1.0

ROOT = Path.cwd().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scripts.parciais as parciais
parciais.HEADERS = globals().get("HEADERS", {})
fetch_pontuados = parciais.fetch_pontuados
fetch_time_payload = parciais.fetch_time_payload
clubes_que_ja_jogaram = parciais.clubes_que_ja_jogaram
calcular_parcial_time = parciais.calcular_parcial_time

sess = requests.Session()
sess.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
})


def http_status_e_rodada():
    r = sess.get("https://api.cartola.globo.com/mercado/status", timeout=20)
    r.raise_for_status()
    data = r.json()
    return int(data.get("status_mercado", 0)), int(data.get("rodada_atual", 0))


def _coerce_float(value):
    try:
        if value is None:
            return None
        return float(value)
    except Exception:
        return None


def extrair_pontuacao_payload(payload):
    if not isinstance(payload, dict):
        return None

    candidatos = []
    for chave in ("pontos", "pontuacao"):
        candidatos.append(payload.get(chave))

    time_info = payload.get("time", {})
    if isinstance(time_info, dict):
        for chave in ("pontos", "pontuacao"):
            candidatos.append(time_info.get(chave))

    pontos_info = payload.get("pontos")
    if isinstance(pontos_info, dict):
        for chave in ("rodada", "total", "valor"):
            candidatos.append(pontos_info.get(chave))

    for candidato in candidatos:
        valor = _coerce_float(candidato)
        if valor is not None:
            return valor

    return None


def obter_pontuacao_rodada_fechada(api, time_id, rodada):
    payload = fetch_time_payload(int(time_id), int(rodada))
    pontuacao_payload = extrair_pontuacao_payload(payload)
    if pontuacao_payload is not None:
        return pontuacao_payload

    time_rodada = api.time(time_id=int(time_id), rodada=int(rodada))
    for atributo in ("pontos", "pontuacao", "ultima_pontuacao"):
        valor = _coerce_float(getattr(time_rodada, atributo, None))
        if valor is not None:
            return valor

    return None


def gerar_df_pontuacoes(api, ids_times):
    global status_http, rodada_http, rodada_api, rod_ref, FASE_FIM, RODADAS_CONCLUIDAS_FIM

    colunas_fase_3 = [f"Rodada {i}" for i in range(FASE_INICIO, FASE_LIMITE + 1)]

    try:
        status_http, rodada_http = http_status_e_rodada()
    except Exception:
        status_http, rodada_http = 0, 0

    try:
        rodada_api = int(api.mercado().rodada_atual)
    except Exception:
        rodada_api = int(rodada_http or 0)

    rod_ref = int(rodada_http or rodada_api or 0)

    if status_http == 2:
        FASE_FIM = max(FASE_INICIO, min(FASE_LIMITE, rod_ref))
        RODADAS_CONCLUIDAS_FIM = FASE_FIM - 1
    elif status_http == 1:
        FASE_FIM = max(FASE_INICIO, min(FASE_LIMITE, max(rod_ref - 1, FASE_INICIO)))
        RODADAS_CONCLUIDAS_FIM = FASE_FIM
    else:
        FASE_FIM = max(FASE_INICIO, min(FASE_LIMITE, rod_ref if rod_ref else FASE_INICIO))
        RODADAS_CONCLUIDAS_FIM = FASE_FIM

    print(
        f"Status={status_http} | rodada_http={rodada_http} | rodada_api={rodada_api} | "
        f"FASE_INICIO={FASE_INICIO} | FASE_FIM={FASE_FIM} | fechadas ate {RODADAS_CONCLUIDAS_FIM}"
    )

    dados = {}
    for nome, time_id in ids_times.items():
        pontuacoes = {}
        if RODADAS_CONCLUIDAS_FIM >= FASE_INICIO:
            for rodada in range(FASE_INICIO, RODADAS_CONCLUIDAS_FIM + 1):
                try:
                    pontuacoes[rodada] = obter_pontuacao_rodada_fechada(api, time_id, rodada)
                except Exception as e:
                    print(f"Erro ao acessar pontuacao da rodada {rodada} para o time {time_id}: {e}")
                    pontuacoes[rodada] = None
        dados[nome] = {f"Rodada {rodada}": pontuacoes.get(rodada) for rodada in range(FASE_INICIO, FASE_LIMITE + 1)}

    df = pd.DataFrame.from_dict(dados, orient="index")
    df = df.reindex(columns=colunas_fase_3)
    df = df.apply(pd.to_numeric, errors="coerce")

    col_atual = f"Rodada {rod_ref}"
    if status_http == 2 and FASE_INICIO <= rod_ref <= FASE_LIMITE:
        print(f"Rodada {rod_ref} em andamento: aplicando parciais")
        mapa_pontuados = fetch_pontuados()
        if mapa_pontuados:
            clubes_jogaram = clubes_que_ja_jogaram(rod_ref)
            for nome_time, time_id in ids_times.items():
                try:
                    total = calcular_parcial_time(int(time_id), rod_ref, mapa_pontuados, clubes_jogaram)
                    if col_atual not in df.columns:
                        df[col_atual] = pd.NA
                    df.loc[nome_time, col_atual] = round(total, 2)
                except Exception as e:
                    print(f"Erro ao calcular parcial da rodada {rod_ref} para o time {time_id}: {e}")
                time.sleep(PER_REQ_SLEEP)
        else:
            print("Parciais indisponiveis no momento.")
    else:
        print("Sem parciais para aplicar nesta fase.")

    colunas_com_dados = [col for col in df.columns if df[col].notna().any()]
    if colunas_com_dados:
        lideres = {col: df[col].dropna().idxmax() for col in colunas_com_dados}
        df = pd.concat([df, pd.DataFrame([lideres], index=["Lider_Rodada"])], axis=0)

    return df


In [13]:
ids_times = {v: k for k, v in nomes_por_id.items()}

df_pontuacoes = gerar_df_pontuacoes(api, ids_times)
display(df_pontuacoes.T)

Status=2 | rodada_http=13 | rodada_api=13 | FASE_INICIO=13 | FASE_FIM=13 | fechadas ate 12
Rodada 13 em andamento: aplicando parciais


,Bandoleros FCS,LISI GREMISTA,cartola scheuer17,Dom Camillo68,TEAM LOPES 99,Fedato Futebol Clube,lsauer fc,Tabajara de Inhaua PB1,Lider_Rodada
Rodada 13,92.62,94.47,97.62,111.22,87.92,136.52,108.52,105.72,Fedato Futebol Clube
Rodada 14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# def classificacao_por_grupo(df_rodadas, df_pontuacoes):

#     df_pontuacoes_times = df_pontuacoes.drop(index='Lider_Rodada', errors='ignore')
#     estatisticas = {}

#     for _, confronto in df_rodadas.iterrows():
#         rodada = confronto["Rodada"]
#         mandante = confronto["Mandante_Nome"]
#         visitante = confronto["Visitante_Nome"]
#         jogo = confronto["Jogo"]
#         coluna_rodada = f"Rodada {rodada}"

#         if mandante not in df_pontuacoes_times.index or visitante not in df_pontuacoes_times.index:
#             continue
#         if coluna_rodada not in df_pontuacoes_times.columns:
#             continue

#         pontos_mandante = df_pontuacoes_times.at[mandante, coluna_rodada]
#         pontos_visitante = df_pontuacoes_times.at[visitante, coluna_rodada]

#         # Ignorar confrontos ainda não disputados (com pontuação 0 ou ausente)
#         if (
#             pd.isnull(pontos_mandante) or pd.isnull(pontos_visitante) or
#             (pontos_mandante == 0 and pontos_visitante == 0)
#         ):
#             continue

#         for time in [mandante, visitante]:
#             if jogo not in estatisticas:
#                 estatisticas[jogo] = {}
#             if time not in estatisticas[jogo]:
#                 estatisticas[jogo][time] = {
#                     "Pontos": 0, "Vitórias": 0, "Empates": 0, "Derrotas": 0,
#                     "Total_Cartola": 0,
#                     "Cartola_Sofrido": 0
#                 }

#         # Atualizar estatísticas do jogo
#         estatisticas[jogo][mandante]["Total_Cartola"] += pontos_mandante
#         estatisticas[jogo][mandante]["Cartola_Sofrido"] += pontos_visitante

#         estatisticas[jogo][visitante]["Total_Cartola"] += pontos_visitante
#         estatisticas[jogo][visitante]["Cartola_Sofrido"] += pontos_mandante

#         if pontos_mandante > pontos_visitante:
#             estatisticas[jogo][mandante]["Pontos"] += 3
#             estatisticas[jogo][mandante]["Vitórias"] += 1
#             estatisticas[jogo][visitante]["Derrotas"] += 1
#         elif pontos_mandante < pontos_visitante:
#             estatisticas[jogo][visitante]["Pontos"] += 3
#             estatisticas[jogo][visitante]["Vitórias"] += 1
#             estatisticas[jogo][mandante]["Derrotas"] += 1
#         else:
#             estatisticas[jogo][mandante]["Pontos"] += 1
#             estatisticas[jogo][visitante]["Pontos"] += 1
#             estatisticas[jogo][mandante]["Empates"] += 1
#             estatisticas[jogo][visitante]["Empates"] += 1

#     # Gerar DataFrame final
#     df_resultado = pd.concat([
#         pd.DataFrame({
#             "Jogo": jogo,
#             "Nome do Time": list(times.keys()),
#             "Pontos": [stats["Pontos"] for stats in times.values()],
#             "Vitórias": [stats["Vitórias"] for stats in times.values()],
#             "Empates": [stats["Empates"] for stats in times.values()],
#             "Derrotas": [stats["Derrotas"] for stats in times.values()],
#             "Total Cartola": [stats["Total_Cartola"] for stats in times.values()],
#             "Cartola Sofrido": [stats["Cartola_Sofrido"] for stats in times.values()],
#             "Saldo Cartola": [
#                 stats["Total_Cartola"] - stats["Cartola_Sofrido"] for stats in times.values()
#             ]
#         }) for jogo, times in estatisticas.items()
#     ], ignore_index=True)

#     # Ordenar e adicionar posição
#     df_resultado = df_resultado.sort_values(
#         by=["Jogo", "Pontos", "Vitórias", "Total Cartola", "Saldo Cartola"],
#         ascending=[True, False, False, False, False]
#     )

#     df_resultado["Posição"] = df_resultado.groupby("Jogo").cumcount() + 1

#     df_resultado_por_jogo = {
#         jogo: df_resultado[df_resultado["Jogo"] == jogo] for jogo in df_resultado["Jogo"].unique()
#     }

#     return df_resultado, df_resultado_por_jogo


In [15]:
def classificacao_por_grupo(df_rodadas, df_pontuacoes):
    """Gera classificação por grupo (ou jogo), exibindo times mesmo com pontuações zeradas."""
    df_pontuacoes_times = df_pontuacoes.drop(index='Lider_Rodada', errors='ignore')
    estatisticas = {}

    for _, confronto in df_rodadas.iterrows():
        rodada = confronto.get("Rodada")
        mandante = confronto.get("Mandante_Nome")
        visitante = confronto.get("Visitante_Nome")
        jogo = confronto.get("Jogo")
        coluna_rodada = f"Rodada {rodada}"

        # Pula se o jogo não existir na planilha
        if mandante not in df_pontuacoes_times.index or visitante not in df_pontuacoes_times.index:
            continue
        if coluna_rodada not in df_pontuacoes_times.columns:
            continue

        pontos_mandante = df_pontuacoes_times.at[mandante, coluna_rodada]
        pontos_visitante = df_pontuacoes_times.at[visitante, coluna_rodada]

        # Se não tiver pontuação ainda, define como 0
        if pd.isnull(pontos_mandante):
            pontos_mandante = 0
        if pd.isnull(pontos_visitante):
            pontos_visitante = 0

        # Inicializa estrutura
        if jogo not in estatisticas:
            estatisticas[jogo] = {}
        for time in [mandante, visitante]:
            if time not in estatisticas[jogo]:
                estatisticas[jogo][time] = {
                    "Pontos": 0, "Vitórias": 0, "Empates": 0, "Derrotas": 0,
                    "Total_Cartola": 0, "Cartola_Sofrido": 0
                }

        # Atualiza totais
        estatisticas[jogo][mandante]["Total_Cartola"] += pontos_mandante
        estatisticas[jogo][mandante]["Cartola_Sofrido"] += pontos_visitante
        estatisticas[jogo][visitante]["Total_Cartola"] += pontos_visitante
        estatisticas[jogo][visitante]["Cartola_Sofrido"] += pontos_mandante

        # Atualiza pontos somente se já houve pontuação real
        if pontos_mandante == 0 and pontos_visitante == 0:
            continue

        if pontos_mandante > pontos_visitante:
            estatisticas[jogo][mandante]["Pontos"] += 3
            estatisticas[jogo][mandante]["Vitórias"] += 1
            estatisticas[jogo][visitante]["Derrotas"] += 1
        elif pontos_mandante < pontos_visitante:
            estatisticas[jogo][visitante]["Pontos"] += 3
            estatisticas[jogo][visitante]["Vitórias"] += 1
            estatisticas[jogo][mandante]["Derrotas"] += 1
        else:
            estatisticas[jogo][mandante]["Pontos"] += 1
            estatisticas[jogo][visitante]["Pontos"] += 1
            estatisticas[jogo][mandante]["Empates"] += 1
            estatisticas[jogo][visitante]["Empates"] += 1

    # 🟡 Caso ainda não existam pontuações, monta estrutura zerada
    if not estatisticas:
        print("📊 Exibindo classificação inicial com pontuações zeradas.")
        jogos = df_rodadas["Jogo"].unique()
        estatisticas = {
            jogo: {
                row["Mandante_Nome"]: {"Pontos": 0, "Vitórias": 0, "Empates": 0, "Derrotas": 0,
                                       "Total_Cartola": 0, "Cartola_Sofrido": 0},
                row["Visitante_Nome"]: {"Pontos": 0, "Vitórias": 0, "Empates": 0, "Derrotas": 0,
                                        "Total_Cartola": 0, "Cartola_Sofrido": 0}
            }
            for jogo, row in df_rodadas.groupby("Jogo").first().iterrows()
        }

    # 🧾 Monta DataFrame final
    df_resultado = pd.concat([
        pd.DataFrame({
            "Jogo": jogo,
            "Nome do Time": list(times.keys()),
            "Pontos": [stats["Pontos"] for stats in times.values()],
            "Vitórias": [stats["Vitórias"] for stats in times.values()],
            "Empates": [stats["Empates"] for stats in times.values()],
            "Derrotas": [stats["Derrotas"] for stats in times.values()],
            "Total Cartola": [stats["Total_Cartola"] for stats in times.values()],
            "Cartola Sofrido": [stats["Cartola_Sofrido"] for stats in times.values()],
            "Saldo Cartola": [
                stats["Total_Cartola"] - stats["Cartola_Sofrido"] for stats in times.values()
            ]
        })
        for jogo, times in estatisticas.items()
    ], ignore_index=True)

    # 🏁 Ordena e adiciona posição
    df_resultado = df_resultado.sort_values(
        by=["Jogo", "Pontos", "Vitórias", "Total Cartola", "Saldo Cartola"],
        ascending=[True, False, False, False, False]
    )
    df_resultado["Posição"] = df_resultado.groupby("Jogo").cumcount() + 1

    df_resultado_por_jogo = {
        jogo: df_resultado[df_resultado["Jogo"] == jogo] for jogo in df_resultado["Jogo"].unique()
    }

    return df_resultado, df_resultado_por_jogo


In [16]:
# Padroniza os nomes no df_rodadas
df_rodadas["Mandante_Nome"] = df_rodadas["Mandante_Nome"].str.strip()
df_rodadas["Visitante_Nome"] = df_rodadas["Visitante_Nome"].str.strip()

# Padroniza os índices do df_pontuacoes
df_pontuacoes.index = df_pontuacoes.index.str.strip()

# Exibe para conferência
display(df_pontuacoes)

,Rodada 13,Rodada 14
Bandoleros FCS,92.62,NaN
LISI GREMISTA,94.47,NaN
cartola scheuer17,97.62,NaN
Dom Camillo68,111.22,NaN
TEAM LOPES 99,87.92,NaN
Fedato Futebol Clube,136.52,NaN
lsauer fc,108.52,NaN
Tabajara de Inhaua PB1,105.72,NaN
Lider_Rodada,Fedato Futebol Clube,NaN


In [17]:
# Gerar a classificação da fase 3
df_resultado, df_resultado_por_jogo = classificacao_por_grupo(
    df_rodadas, df_pontuacoes
)

# Salvar cada grupo em uma aba do Excel
with pd.ExcelWriter("classificacao_por_grupo_fase_3.xlsx") as writer:
    for jogo, df in df_resultado_por_jogo.items():
        df.to_excel(writer, sheet_name=jogo, index=False)

# Exibir a classificação geral
df_resultado_jogo_1 = df_resultado[df_resultado["Jogo"] == "Jogo 1 (JG1)"]
df_resultado_jogo_2 = df_resultado[df_resultado["Jogo"] == "Jogo 2 (JG2)"]
df_resultado_jogo_3 = df_resultado[df_resultado["Jogo"] == "Jogo 3 (JG3)"]
df_resultado_jogo_4 = df_resultado[df_resultado["Jogo"] == "Jogo 4 (JG4)"]

display(df_resultado_jogo_1, df_resultado_jogo_2, df_resultado_jogo_3, df_resultado_jogo_4)


,Jogo,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola,Posição
0,Jogo 1 (JG1),lsauer fc,3,1,0,0,108.52,105.72,2.80,1
1,Jogo 1 (JG1),Tabajara de Inhaua PB1,0,0,0,1,105.72,108.52,-2.80,2


,Jogo,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola,Posição
3,Jogo 2 (JG2),LISI GREMISTA,3,1,0,0,94.47,92.62,1.85,1
2,Jogo 2 (JG2),Bandoleros FCS,0,0,0,1,92.62,94.47,-1.85,2


,Jogo,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola,Posição
4,Jogo 3 (JG3),Fedato Futebol Clube,3,1,0,0,136.52,111.22,25.30,1
5,Jogo 3 (JG3),Dom Camillo68,0,0,0,1,111.22,136.52,-25.30,2


,Jogo,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola,Posição
7,Jogo 4 (JG4),cartola scheuer17,3,1,0,0,97.62,87.92,9.70,1
6,Jogo 4 (JG4),TEAM LOPES 99,0,0,0,1,87.92,97.62,-9.70,2


In [18]:
# Verifica se a variável df_resultado_por_grupo existe e está populada
if 'df_resultado_por_jogo' in locals() and df_resultado_por_jogo:
    # Criar estrutura em formato de dicionário para JSON/JS
    classificacao_js_fase_3 = {}

    for jogo, df in df_resultado_por_jogo.items():
        classificacao_js_fase_3[jogo] = []
        for _, row in df.iterrows():
            classificacao_js_fase_3[jogo].append({
                "posicao": int(row["Posição"]),
                "nome": row["Nome do Time"],
                "pontos": int(row["Pontos"]),
                "vitorias": int(row["Vitórias"]),
                "empates": int(row["Empates"]),
                "derrotas": int(row["Derrotas"]),
                "totalCartola": float(row["Total Cartola"]),
                "cartolaSofrido": float(row["Cartola Sofrido"]),
                "saldoCartola": float(row["Saldo Cartola"])
            })

    # Converter para JSON formatado
    json_str = json.dumps(classificacao_js_fase_3, indent=2, ensure_ascii=False)

    # Salvar como arquivo JS com uma variável global
    with open("classificacao_por_grupo_fase_3.js", "w", encoding="utf-8") as f:
        f.write("const classificacaoFase3 = ")
        f.write(json_str)
        f.write(";")

    print("✅ Arquivo JS salvo com sucesso.")

else:
    print("⚠️ Classificação da fase 3 indisponível. O mercado pode estar em manutenção ou os dados ainda não foram gerados.")

✅ Arquivo JS salvo com sucesso.


In [19]:
def exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada, jogo=None):
    """
    Exibe os resultados de uma rodada específica, com pontuação e dados dos times.
    """

    if rodada not in df_rodadas["Rodada"].values:
        return pd.DataFrame([{
            "Jogo": jogo or "-",
            "Rodada": rodada,
            "Mandante_Nome": "-",          
            "Mandante_Pontos": "-",
            "Visitante_Nome": "-",           
            "Visitante_Pontos": "-",
        }])

    df_filtrado = df_rodadas[df_rodadas["Rodada"] == rodada]
    if jogo:
        df_filtrado = df_filtrado[df_filtrado["Jogo"] == jogo]

    resultados = []

    for _, row in df_filtrado.iterrows():
        jogo_ = row["Jogo"]
        mandante = row["Mandante_Nome"]
        visitante = row["Visitante_Nome"]

        pontos_mandante = df_pontuacoes.get(f"Rodada {rodada}", {}).get(mandante, None)
        pontos_visitante = df_pontuacoes.get(f"Rodada {rodada}", {}).get(visitante, None)

        resultados.append({
            "Jogo": jogo_,
            "Rodada": rodada,
            "Mandante_Nome": mandante,
            "Mandante_Pontos": pontos_mandante,
            "Visitante_Nome": visitante,
            "Visitante_Pontos": pontos_visitante
        })

    return pd.DataFrame(resultados)


In [20]:
# Exibir resultados da 13a rodada
df_resultados_rodada_13 = exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada=13)

# Exibir apenas os resultados do Grupo B na 1ª rodada
df_resultados_jogo_1 = exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada=13, jogo="Jogo 1 (JG1)")

# Exibir
display(df_resultados_rodada_13)

,Jogo,Rodada,Mandante_Nome,Mandante_Pontos,Visitante_Nome,Visitante_Pontos
0,Jogo 1 (JG1),13,lsauer fc,108.52,Tabajara de Inhaua PB1,105.72
1,Jogo 2 (JG2),13,Bandoleros FCS,92.62,LISI GREMISTA,94.47
2,Jogo 3 (JG3),13,Fedato Futebol Clube,136.52,Dom Camillo68,111.22
3,Jogo 4 (JG4),13,TEAM LOPES 99,87.92,cartola scheuer17,97.62


In [21]:
# Criar arquivo com uma aba para cada rodada contendo os resultados detalhados
from pathlib import Path

# Caminho do arquivo de saída
caminho_resultados = "resultados_fase_3.xlsx"

# Descobrir as rodadas únicas no DataFrame
rodadas_disponiveis = sorted(df_rodadas["Rodada"].unique())

with pd.ExcelWriter(caminho_resultados) as writer:
    for rodada in rodadas_disponiveis:
        df_resultados = exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada=rodada)
        nome_aba = f"Rodada {rodada}"
        df_resultados.to_excel(writer, sheet_name=nome_aba, index=False)

print(f"Arquivo salvo com sucesso: {Path(caminho_resultados).resolve()}")

display(df_resultados)

Arquivo salvo com sucesso: C:\Users\ferna\Projetos\GitHub\cartola_2026\libertadores\datasets_liberta\resultados_fase_3.xlsx


,Jogo,Rodada,Mandante_Nome,Mandante_Pontos,Visitante_Nome,Visitante_Pontos
0,Jogo 1 (JG1),14,Tabajara de Inhaua PB1,NaN,lsauer fc,NaN
1,Jogo 2 (JG2),14,LISI GREMISTA,NaN,Bandoleros FCS,NaN
2,Jogo 3 (JG3),14,Dom Camillo68,NaN,Fedato Futebol Clube,NaN
3,Jogo 4 (JG4),14,cartola scheuer17,NaN,TEAM LOPES 99,NaN


In [22]:
resultados_js = []

for rodada in sorted(df_rodadas["Rodada"].unique()):
    df_resultados = exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada=rodada)
    
    for _, row in df_resultados.iterrows():
        resultado = {
            "jogo": row["Jogo"],
            "rodada": int(rodada),
            "mandante": {
                "nome": row["Mandante_Nome"],
                "pontos": float(row["Mandante_Pontos"]) if row["Mandante_Pontos"] is not None else None
            },
            "visitante": {
                "nome": row["Visitante_Nome"],
                "pontos": float(row["Visitante_Pontos"]) if row["Visitante_Pontos"] is not None else None
            },
            "vencedor": (
                "mandante" if row["Mandante_Pontos"] is not None and row["Visitante_Pontos"] is not None and row["Mandante_Pontos"] > row["Visitante_Pontos"]
                else "visitante" if row["Mandante_Pontos"] is not None and row["Visitante_Pontos"] is not None and row["Mandante_Pontos"] < row["Visitante_Pontos"]
                else "empate" if row["Mandante_Pontos"] == row["Visitante_Pontos"] and row["Mandante_Pontos"] is not None
                else "indefinido"
            )

        }
        resultados_js.append(resultado)

# Exportar para arquivo .js
import json

with open("resultados_fase_3.js", "w", encoding="utf-8") as f:
    f.write("const resultadosFase3 = ")
    f.write(json.dumps(resultados_js, indent=2, ensure_ascii=False))
    f.write(";")

# Exporta meta de parcial (usada no front)
try:
    rodada_ref = int(rod_ref)
except Exception:
    rodada_ref = 0

parcial_payload = {"rodada": rodada_ref, "times": {}}
try:
    col_parcial = f"Rodada {rodada_ref}"
    if col_parcial in df_pontuacoes.columns:
        times_map = {}
        for nome in df_pontuacoes.index:
            if nome not in ids_times:
                continue
            try:
                val = df_pontuacoes.at[nome, col_parcial]
            except Exception:
                continue
            if str(val) in ("", "nan"):
                continue
            try:
                times_map[str(ids_times[nome])] = float(val)
            except Exception:
                continue
        parcial_payload["times"] = times_map
except Exception:
    pass

try:
    status_http_val = int(status_http)
except Exception:
    status_http_val = None

liberta_meta = {
    "rodada_atual": rodada_ref,
    "parcial_disponivel": bool(status_http_val == 2 and parcial_payload["times"])
}

with open("resultados_fase_3.js", "a", encoding="utf-8") as f:
    f.write("const pontuacaoParcialRodadaAtual = ")
    f.write(json.dumps(parcial_payload, indent=2, ensure_ascii=False))
    f.write(";")
    f.write("window.libertaMeta = ")
    f.write(json.dumps(liberta_meta, indent=2, ensure_ascii=False))
    f.write(";")



### Identificando Vencedores das Quartas de Final

In [23]:
def obter_classificados_com_id(resultados, ids_por_nome):
    classificados = []
    jogos_agrupados = {}
    for jogo in resultados:
        chave = jogo['jogo']
        if chave not in jogos_agrupados:
            jogos_agrupados[chave] = []
        jogos_agrupados[chave].append(jogo)

    for chave, partidas in jogos_agrupados.items():
        if len(partidas) < 2:
            continue

        partidas = sorted(partidas, key=lambda x: x['rodada'])
        ida, volta = partidas

        if ida['mandante']['pontos'] is None or volta['mandante']['pontos'] is None:
            continue

        time1 = ida['mandante']['nome']
        time2 = ida['visitante']['nome']

        pontos_time1 = ida['mandante']['pontos'] + volta['visitante']['pontos']
        pontos_time2 = ida['visitante']['pontos'] + volta['mandante']['pontos']

        if pontos_time1 > pontos_time2:
            vencedor = time1
        elif pontos_time2 > pontos_time1:
            vencedor = time2
        else:
            vencedor = "EMPATE"

        classificados.append({
            "jogo": chave,
            "classificado_nome": vencedor,
            "classificado_id": ids_por_nome.get(vencedor) if vencedor in ids_por_nome else None
        })

    return classificados


In [24]:
# Inverter os nomes
ids_por_nome = {v: k for k, v in nomes_por_id.items()}

# Ler o resultados_fase_3.js (como já fizemos antes)
resultados = carregar_payload_js("resultados_fase_3.js", "resultadosFase3")

# Obter os classificados
classificados = obter_classificados_com_id(resultados, ids_por_nome)

# Salvar
df_classificados = pd.DataFrame(classificados)
df_classificados.to_excel("classificados_fase_3.xlsx", index=False)

with open("classificados_fase_3.js", "w", encoding="utf-8") as f:
    f.write("const classificadosFase3 = ")
    json.dump(classificados, f, ensure_ascii=False, indent=2)
    f.write(";")
print("✅ Classificados salvos com sucesso em 'classificados_fase_3.xlsx' e 'classificados_fase_3.js'.")

# Exibir para validar
print("Classificados encontrados:", classificados)


✅ Classificados salvos com sucesso em 'classificados_fase_3.xlsx' e 'classificados_fase_3.js'.
Classificados encontrados: [{'jogo': 'Jogo 1 (JG1)', 'classificado_nome': 'EMPATE', 'classificado_id': None}, {'jogo': 'Jogo 2 (JG2)', 'classificado_nome': 'EMPATE', 'classificado_id': None}, {'jogo': 'Jogo 3 (JG3)', 'classificado_nome': 'EMPATE', 'classificado_id': None}, {'jogo': 'Jogo 4 (JG4)', 'classificado_nome': 'EMPATE', 'classificado_id': None}]
